# PrimeKV — Weekend Notebook

One notebook that tests the three things we could not validate on Day 1:

1. **SpaCy classifier** — does a linguistically-aware classifier beat the rule-based one? (gating experiment)
2. **Dynamic tuning** — do policies auto-adapt as memory pressure changes?
3. **Reasoning eval** — does PrimeKV preserve constraints through filler that H2O / StreamingLLM drop?

Target runtime: Colab A100 or L4. Qwen2.5-3B-Instruct on GPU.

Run top-to-bottom.

## 1. Install

Clones the repo, installs the package, downloads the spaCy English model.

In [ ]:
!rm -rf /content/PrimeKV
!git clone --branch primekv2 https://github.com/arunvenkatadri/PrimeKV.git /content/PrimeKV
%cd /content/PrimeKV
!pip install -q -e .[spacy,web]
!python -m spacy download en_core_web_sm

## 2. Load Qwen2.5-3B-Instruct on GPU

Requires `attn_implementation="eager"` so `past_key_values` stays in the standard tuple layout the PrimeKV adapter expects.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Change runtime type to GPU (A100/L4 recommended)'
print('GPU:', torch.cuda.get_device_name(0))

from transformers import AutoModelForCausalLM, AutoTokenizer
MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    attn_implementation='eager',
).to('cuda').eval()
print('loaded', MODEL_NAME, '—', model.config.num_hidden_layers, 'layers,',
      model.config.num_attention_heads, 'heads,',
      model.config.hidden_size // model.config.num_attention_heads, 'head_dim')

# A realistic long-context prompt for all downstream experiments.
LONG_PROMPT = (
    'The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. '
    'It is named after the engineer Gustave Eiffel, whose company designed and built the tower '
    'from 1887 to 1889. Locally nicknamed La dame de fer, it was constructed as the centerpiece '
    'of the 1889 Worlds Fair, and to crown the 100th anniversary of the French Revolution. '
    'Although initially criticised by some of Frances leading artists and intellectuals for its '
    'design, it has since become a global cultural icon of France and one of the most recognisable '
    'structures in the world. '
    'The tower is 330 metres tall, about the same height as an 81-storey building, and was the '
    'tallest man-made structure in the world for 41 years until the Chrysler Building in New York '
    'City was finished in 1930. It was the first structure in the world to surpass both the 200-metre '
    'and 300-metre marks. Due to the addition of a broadcasting aerial at the top of the tower in '
    '1957, it is now taller than the Chrysler Building by 5.2 metres. Excluding transmitters, the '
    'Eiffel Tower is the second tallest free-standing structure in France after the Millau Viaduct. '
) * 3
print('prompt length (chars):', len(LONG_PROMPT))
print('prompt length (tokens):', len(tok(LONG_PROMPT)['input_ids']))

## 3. SpaCy classifier sweep — the gating experiment

Does a classifier that uses actual POS tags / named entities beat a positional stride? If rule-based wins here, the structural-role thesis is not validated yet.

We compare three PrimeKV variants (rule-based vs spaCy) against the full cache and two baselines at matched memory.

In [ ]:
import spacy
from primekv.cache import PrimeKVCache, TierPolicy, DEFAULT_POLICIES
from primekv.classifier import RuleBasedClassifier, SpaCyClassifier, Tier
from primekv.baselines import FullCache, H2OCache, StreamingLLMCache, UniformQuantCache
from primekv.eval import Workload, run_comparison

nlp = spacy.load('en_core_web_sm')
NUM_LAYERS = model.config.num_hidden_layers

def make_primekv(classifier):
    return PrimeKVCache(
        num_layers=NUM_LAYERS,
        classifier=classifier,
        policies=dict(DEFAULT_POLICIES),
        device='cuda',
    )

caches = {
    'full':            FullCache(num_layers=NUM_LAYERS),
    'primekv_rule':    make_primekv(RuleBasedClassifier(anchor_prefix_len=4, semantic_stride=3)),
    'primekv_spacy':   make_primekv(SpaCyClassifier(nlp=nlp, anchor_prefix_len=4)),
    'h2o':             H2OCache(num_layers=NUM_LAYERS, capacity=256),
    'streaming':       StreamingLLMCache(num_layers=NUM_LAYERS, num_sinks=4, window=256),
    'uniform_int4':    UniformQuantCache(num_layers=NUM_LAYERS, bits=4),
}

workload = Workload(prompt=LONG_PROMPT, decode_tokens=32, max_length=3072, name='spacy_gate')
report = run_comparison(caches, workload, model, tok, device='cuda')
print(report.to_markdown())

## 4. Dynamic tuning sweep

Vary the memory budget; watch `auto_tune` pick different operating points and `build_cache_from_profile` materialize each. The perplexity of each profile tells us how well the method degrades under pressure.

In [ ]:
from primekv.tuning import auto_tune, build_cache_from_profile, describe_profile

prompt_len = len(tok(LONG_PROMPT)['input_ids'])
budgets = [None, 128.0, 32.0, 8.0, 2.0]  # None = pick by length alone

tuning_caches = {'full': FullCache(num_layers=NUM_LAYERS)}
profiles = {}
for b in budgets:
    profile = auto_tune(
        prompt_length=prompt_len,
        memory_budget_mb=b,
        num_layers=NUM_LAYERS,
        num_heads=model.config.num_attention_heads,
        head_dim=model.config.hidden_size // model.config.num_attention_heads,
    )
    label = f'tuned_{b if b is None else int(b)}mb'
    profiles[label] = profile
    tuning_caches[label] = build_cache_from_profile(
        profile, num_layers=NUM_LAYERS, device='cuda'
    )

for label, p in profiles.items():
    print(f'=== {label} ===')
    print(describe_profile(p))
    print()

tuning_report = run_comparison(tuning_caches, workload, model, tok, device='cuda')
print(tuning_report.to_markdown())

## 5. Reasoning eval — constraint persistence

This is the test perplexity can't see: does the cache preserve a rule ("never mention blue") across a long filler section? Pass/fail on each of 4 canonical tests.

In [ ]:
from primekv.reasoning_eval import default_tests, run_reasoning_eval

reasoning_caches = {
    'full':          FullCache(num_layers=NUM_LAYERS),
    'primekv_spacy': make_primekv(SpaCyClassifier(nlp=nlp, anchor_prefix_len=4)),
    'primekv_rule':  make_primekv(RuleBasedClassifier(anchor_prefix_len=4, semantic_stride=3)),
    'h2o':           H2OCache(num_layers=NUM_LAYERS, capacity=128),
    'streaming':     StreamingLLMCache(num_layers=NUM_LAYERS, num_sinks=4, window=128),
}

reasoning_report = run_reasoning_eval(
    reasoning_caches, model, tok, tests=default_tests(), device='cuda', max_length=3072,
)
print(reasoning_report.to_markdown())
print()
print('Pass rate per cache:', reasoning_report.pass_rate_per_cache())

In [ ]:
# Per-test, per-cache debug view — useful for figuring out *why* a cache failed.
for r in reasoning_report.results:
    tail = (r.generated or '').rsplit('Answer:', 1)[-1].strip().replace('\n', ' ')[:100]
    print(f"[{r.cache:15s}] {r.test:20s} {'PASS' if r.passed else 'FAIL'}  ({r.reason})")
    print(f"{'':19s}  → {tail}")

## 6. Headline figure — 2D Pareto with SpaCy classifier

The 2D eviction × quantization sweep from Day 1, re-run with the SpaCy classifier instead of rule-based. If SpaCy shifts the PrimeKV curve left (lower PPL at same compression) vs rule-based, the thesis has support.

In [ ]:
from primekv.sweep import sweep_2d_tradeoff, plot_report

caps = [32, 64, 128, 256, 512]
precisions = ['fp16', 'int8', 'int4']

# Inject a SpaCy classifier into PrimeKV for every cell in the sweep.
def spacy_classifier_factory():
    return SpaCyClassifier(nlp=nlp, anchor_prefix_len=4)

report_2d = sweep_2d_tradeoff(
    model=model,
    tokenizer=tok,
    prompt=LONG_PROMPT,
    eviction_caps=caps,
    precisions=precisions,
    decode_tokens=16,
    max_length=2048,
    device='cuda',
    primekv_classifier_factory=spacy_classifier_factory,
)

import matplotlib.pyplot as plt
fig = plot_report(report_2d, output_path='/content/PrimeKV/weekend_2d_spacy.png')
plt.show()

## 7. (Optional) Autoresearch — Karpathy-style overnight loop

Requires `ANTHROPIC_API_KEY` in the env. Runs `--rounds N` of read-propose-evaluate-keep against `autoresearch/experiment.py`. The evaluator uses tiny-gpt2 for speed, so one round is ~30s.

Skip this cell if you don't have an API key.

In [ ]:
%cd /content/PrimeKV
import os
if not os.environ.get('ANTHROPIC_API_KEY'):
    print('Set ANTHROPIC_API_KEY to run this cell.')
else:
    !pip install -q anthropic
    !python -m autoresearch.run --agent anthropic --rounds 3 --device cpu

## Takeaways

Record honest observations here. Fill in after the cells have run:

- **SpaCy vs rule-based** (section 3): ___
- **Tuning adapts under pressure** (section 4): ___
- **Reasoning pass rates** (section 5): ___
- **2D Pareto movement** (section 6): ___